# Data modeling 1: grain, facts and dimensions

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/data-modeling/data-modeling-01-grain-facts-and-dimensions.ipynb)

Every analytics model is built from two kinds of table. **Fact tables** record what happened, one row per
event or measurement, with the numbers you add up. **Dimension tables** describe the things involved (the
customer, the product, the day) with the attributes you filter and group by. Before either exists you decide
the **grain**: what exactly one row of the fact table represents.

This lesson builds the shop's first fact table at the right grain, shows which measures can be added up and
which cannot, and covers the four kinds of fact table you will meet in practice.

**Prerequisites:** [Data modeling 0: start here](data-modeling-00-start-here.ipynb).

*Last reviewed: November 2026, against DuckDB 2.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q duckdb~=2.0

## Setup and sample data

The next cell creates an in-memory DuckDB database and loads `src_customers` (the CRM's current view of each customer), `src_products` (the product catalogue), `src_orders` (one row per order, with fulfilment timestamps), `src_order_lines` (one row per product on an order) and `src_promotions` and `src_order_promotions` (promotions applied to orders): extracts from a small online shop that sells coffee, tea and kitchenware. Every lesson starts from these source tables; [Data modeling 0](data-modeling-00-start-here.ipynb) describes them.

In [ ]:
import duckdb

con = duckdb.connect()  # an in-memory database: the warehouse for this notebook
con.sql("""
create table src_customers (
    customer_id int primary key, name varchar, email varchar, city varchar, segment varchar, created_on date
);
insert into src_customers values
    (101, 'Acme Corp',     'ops@acme.example',     'Leeds', 'smb',        '2025-11-02'),
    (102, 'Bright Retail', 'buy@bright.example',   'York',  'enterprise', '2025-11-20'),  -- was smb until Feb 1
    (103, 'Cyan Traders',  'hi@cyan.example',      'Bath',  'enterprise', '2025-12-05'),
    (104, 'Delta Grocers', 'orders@delta.example', 'Leeds', 'smb',        '2026-01-08'),  -- moved from Hull on Feb 15
    (105, 'Echo Supplies', null,                   'Derby', 'smb',        '2026-01-19'),
    (106, 'Foxtrot Foods', 'team@foxtrot.example', 'York',  'mid-market', '2026-02-10');  -- has not ordered yet
create table src_products (
    sku varchar primary key, product_name varchar, category varchar, department varchar, list_price decimal(8, 2)
);
insert into src_products values
    ('SKU-01', 'House Blend 1kg',    'Coffee',  'Grocery', 18.00),
    ('SKU-02', 'Espresso Beans 1kg', 'Coffee',  'Grocery', 22.00),
    ('SKU-03', 'Green Tea 100 bags', 'Tea',     'Grocery',  6.50),
    ('SKU-04', 'Earl Grey 100 bags', 'Tea',     'Grocery',  5.50),
    ('SKU-05', 'Stoneware Mug',      'Mugs',    'Kitchen',  9.00),
    ('SKU-06', 'Travel Mug',         'Mugs',    'Kitchen', 15.00),
    ('SKU-07', 'Gooseneck Kettle',   'Kettles', 'Kitchen', 65.00),
    ('SKU-08', 'Electric Kettle',    'Kettles', 'Kitchen', 45.00);
create table src_orders (
    order_id int primary key, customer_id int, order_ts timestamp, channel varchar, status varchar,
    shipping_fee decimal(8, 2), shipped_ts timestamp, delivered_ts timestamp
);
insert into src_orders values
    (1001, 101, '2026-01-05 09:12', 'web',   'delivered', 4.99, '2026-01-06 10:00', '2026-01-08 15:00'),
    (1002, 102, '2026-01-09 14:30', 'store', 'delivered', 0.00, '2026-01-09 14:30', '2026-01-09 14:30'),
    (1003, 103, '2026-01-14 11:05', 'web',   'delivered', 0.00, '2026-01-15 09:00', '2026-01-19 12:00'),
    (1004, 101, '2026-01-28 16:40', 'web',   'delivered', 4.99, '2026-01-29 11:00', '2026-02-02 10:30'),
    (1005, 104, '2026-02-03 10:20', 'web',   'delivered', 4.99, '2026-02-04 08:45', '2026-02-06 16:10'),
    (1006, 102, '2026-02-11 13:15', 'web',   'shipped',   0.00, '2026-02-12 09:30', null),
    (1007, 105, '2026-02-14 09:50', 'store', 'delivered', 0.00, '2026-02-14 09:50', '2026-02-14 09:50'),
    (1008, 104, '2026-02-20 18:05', 'web',   'placed',    4.99, null,               null),
    (1009, 103, '2026-02-24 12:00', 'web',   'cancelled', 4.99, null,               null),
    (1010, 101, '2026-02-27 08:30', 'store', 'delivered', 0.00, '2026-02-27 08:30', '2026-02-27 08:30');
create table src_order_lines (
    order_id int, line_no int, sku varchar, qty int, unit_price decimal(8, 2), discount decimal(8, 2),
    primary key (order_id, line_no)
);
insert into src_order_lines values
    (1001, 1, 'SKU-01',  2, 18.00,  0.00), (1001, 2, 'SKU-05', 4,  9.00, 0.00),
    (1002, 1, 'SKU-07',  1, 65.00,  5.00),
    (1003, 1, 'SKU-02', 10, 22.00, 20.00), (1003, 2, 'SKU-03', 5,  6.50, 0.00),
    (1004, 1, 'SKU-06',  2, 15.00,  0.00),
    (1005, 1, 'SKU-01',  3, 18.00,  0.00), (1005, 2, 'SKU-04', 2,  5.50, 0.00), (1005, 3, 'SKU-08', 1, 45.00, 0.00),
    (1006, 1, 'SKU-02', 20, 22.00, 40.00),
    (1007, 1, 'SKU-05',  6,  9.00,  0.00),
    (1008, 1, 'SKU-03',  4,  6.50,  0.00),
    (1009, 1, 'SKU-07',  1, 65.00,  0.00),
    (1010, 1, 'SKU-01',  1, 18.00,  0.00), (1010, 2, 'SKU-06', 1, 15.00, 0.00);
create table src_promotions (promo_code varchar primary key, promo_name varchar);
insert into src_promotions values ('WINTER10', 'Winter sale'), ('BULK', 'Bulk discount'), ('LOYAL', 'Loyalty reward');
create table src_order_promotions (order_id int, promo_code varchar, primary key (order_id, promo_code));
insert into src_order_promotions values (1002, 'WINTER10'), (1003, 'BULK'), (1003, 'LOYAL'), (1006, 'BULK');
""")
con.sql("show tables").show()

## Start from the process and the grain

Kimball's design method has four steps, in this order:

1. **Pick the business process**: an activity that produces events, such as taking orders, receiving
   payments or shipping parcels. Not a department and not a report.
2. **Declare the grain**: one sentence that says what one row is. "One row per product on an order" is a
   grain. "Order data" is not.
3. **Choose the dimensions**: the descriptions that are true for one row at that grain (which customer,
   which product, which day, which channel).
4. **Choose the facts**: the numbers measured at that grain (quantity, amount, discount).

The grain comes before any column because it decides which questions the table can answer. A table at
order grain cannot say how much coffee was sold, because an order mixes products. A table at order-line
grain can answer that *and* still roll up to orders. So the default is the **atomic grain**: the finest
detail the source records. You can always aggregate a fine table; you can never split a coarse one.

The grain is also the table's key. "One row per product on an order" means `(order_id, line_no)` is unique.
If the key you expect is not unique, the grain you declared is not the grain you have.

**Where it matters**
- Every new fact table: write the grain sentence in its description before writing SQL.
- Reviews: most wrong numbers trace back to a table whose real grain differs from what its users assume.

**Watch out for**
- Starting from a report ("the monthly sales dashboard") instead of a process: you build a coarse table that answers one question.
- A grain sentence that hides two grains, such as orders *and* their lines in one table.

### Example

The order table is at order grain, so it has no product. Here is the obvious attempt at "sales by category" from
it, and the fact table at order-line grain that can answer it.

In [ ]:
con.sql("""
select column_name, column_type from (describe src_orders)
""").show()

In [ ]:
con.sql("""
create table fact_order_line as
select l.order_id,                                    -- which order (kept on the fact; see below)
       l.line_no,
       o.customer_id,
       l.sku,
       cast(o.order_ts as date)                as order_date,
       o.channel,
       l.qty                                   as quantity,
       l.qty * l.unit_price                    as gross_amount,
       l.discount                              as discount_amount,
       l.qty * l.unit_price - l.discount       as net_amount
from src_order_lines l
join src_orders o using (order_id)
where o.status <> 'cancelled'
""")
con.sql("select * from fact_order_line order by order_id, line_no").show()

In [ ]:
con.sql("""
-- the grain sentence as a check: one row per (order_id, line_no)
select count(*) as fact_rows, count(distinct (order_id, line_no)) as distinct_keys from fact_order_line
""").show()

**What to notice:** `src_orders` has no `sku` column, so no query on it can split sales by product. `fact_order_line` has 14 rows,
one per line of a non-cancelled order, and 14 distinct `(order_id, line_no)` keys: the declared grain holds.
Order 1009 is gone because it was cancelled; whether cancelled orders belong in a sales fact is a business
rule to agree on and write down, not a technical detail.

## Facts and dimensions

A **fact** is a measurement taken at the grain: `quantity`, `net_amount`. Facts are usually numeric and you
aggregate them. A **dimension** is the context of the measurement: the customer's segment, the product's
category, the date. You filter and group by dimension attributes.

The fact table holds the measures and a key to each dimension. It is tall and narrow and grows with every
event. Dimension tables are short and wide: one row per customer or product with as many descriptive columns
as are useful. Joined together they form a **star**: the fact in the middle, a dimension at each point.

Some context has no attributes beyond its identifier. `order_id` on `fact_order_line` is like that: it
groups lines into orders, but everything else about the order (date, channel, customer) is already on the
row. Kimball calls it a **degenerate dimension**: a dimension key with no dimension table.

Each fact-to-dimension join must be **many-to-one**: many fact rows to exactly one dimension row. Then the
join only adds columns. It never adds or removes fact rows, so totals do not change.

**Where it matters**
- Deciding where a column goes: if it describes a thing and changes rarely, it belongs on a dimension; if it is measured per event, on the fact.
- BI tools: they expect exactly this shape, with measures on facts and slicers from dimensions.

**Watch out for**
- Descriptive text copied onto the fact (product name, customer city). It bloats the table and goes stale when the description changes.
- A dimension with duplicate keys. The join becomes many-to-many and silently multiplies fact rows (lesson 4).

### Example

Build two simple dimensions and use them to slice the fact.

In [ ]:
con.sql("""
create table dim_product as
select sku, product_name, category, department from src_products;

create table dim_customer as
select customer_id, name as customer_name, city, segment from src_customers;
""")
con.sql("""
select p.department, p.category, c.segment, sum(f.quantity) as units, sum(f.net_amount) as net_sales
from fact_order_line f
join dim_product p using (sku)
join dim_customer c using (customer_id)
group by all
order by all
""").show()

In [ ]:
con.sql("""
select (select sum(net_amount) from fact_order_line) as fact_total,
       sum(f.net_amount)                             as total_after_joins
from fact_order_line f
join dim_product p using (sku)
join dim_customer c using (customer_id)
""").show()

**What to notice:** The total is 1,017.50 before and after joining both dimensions. Each fact row found exactly one product and one
customer, so the joins added columns without changing the row count. That is the property to protect: if a
dimension ever has two rows for one key, this check is the first thing to break.

## Additive, semi-additive and non-additive measures

Whether a measure can be summed depends on what it measures.

- **Additive** measures can be summed across every dimension: `quantity`, `net_amount`. Sales on Monday plus
  sales on Tuesday is sales for both days.
- **Semi-additive** measures can be summed across some dimensions but not time. Stock on hand and account
  balances are levels, not flows: stock of 10 on Monday and 10 on Tuesday is not 20 units. Across time take
  the closing value (or an average); across products and stores, sum.
- **Non-additive** measures cannot be summed at all: ratios, percentages, averages, distinct counts. Store
  their *components* (revenue and order count) and divide after aggregating.

Record which kind each measure is, next to the measure. A BI tool will happily sum anything.

**Where it matters**
- Inventory, account balances, headcount, open tickets: all semi-additive levels.
- Margins, conversion rates and average order value: compute from summed components at query time.

**Watch out for**
- Averaging an average: a day with one order weighs as much as a day with a hundred.
- Summing distinct counts across groups: a customer active in two months is counted twice.

### Example

A daily stock snapshot is semi-additive. The cell creates three days of stock for two products and compares the
wrong and right ways to report "stock at the end of the period".

In [ ]:
con.sql("""
create table fact_stock_daily (snapshot_date date, sku varchar, units_on_hand int);
insert into fact_stock_daily values
    ('2026-02-26', 'SKU-01', 40), ('2026-02-26', 'SKU-05', 25),
    ('2026-02-27', 'SKU-01', 39), ('2026-02-27', 'SKU-05', 25),
    ('2026-02-28', 'SKU-01', 39), ('2026-02-28', 'SKU-05', 19);
""")
con.sql("""
select sku,
       sum(units_on_hand)                                     as wrong_summed_over_days,
       arg_max(units_on_hand, snapshot_date)                  as closing_stock,
       round(avg(units_on_hand), 1)                           as average_stock
from fact_stock_daily
group by sku
order by sku
""").show()

**What to notice:** Summing over three days reports 118 units of SKU-01 when the warehouse never held more than 40. The closing
value (`arg_max` picks the value on the latest date) is 39, and the average of 39.3 is the other legitimate
answer. Summing across products on one day is fine: on 2026-02-28 the shop held 39 + 19 = 58 units.

### Example

Average order value is non-additive. Compare averaging monthly averages with dividing totals.

In [ ]:
con.sql("""
with orders as (
    select order_id, date_trunc('month', order_date) as month, sum(net_amount) as order_value
    from fact_order_line group by all
),
monthly as (
    select month, count(*) as orders, sum(order_value) as revenue, avg(order_value) as aov
    from orders group by month
)
select round(avg(aov), 2)                      as wrong_average_of_monthly_aov,
       round(sum(revenue) / sum(orders), 2)    as right_revenue_over_orders,
       list(month::date || ': ' || orders || ' orders' order by month) as months
from monthly
""").show()

**What to notice:** January has 4 orders and February 5, with different average sizes. Averaging the two monthly averages gives
each month equal weight and lands on 111.61; the true average over all 9 orders is 1,017.50 / 9 = 113.06. Keep
`revenue` and `orders` as additive facts and divide at the end.

## The four kinds of fact table

| Kind | One row per | Rows change? | Answers |
|---|---|---|---|
| **Transaction** | event (an order line, a payment) | no, insert only | how many, how much, when |
| **Periodic snapshot** | thing per period (stock per product per day) | no, a new row each period | what was the level on a date |
| **Accumulating snapshot** | instance of a process with fixed steps (an order from placed to delivered) | yes, updated as each step completes | how long each step took, what is stuck |
| **Factless** | event or relationship with nothing to measure (a promotion applied to an order) | no | what happened, or what did not |

`fact_order_line` is a transaction fact and `fact_stock_daily` a periodic snapshot. An **accumulating
snapshot** has one date column per milestone and the lags between them; it is the only kind whose rows are
updated after they are written, and the right model for pipelines with a defined path: fulfilment, claims,
loan applications. A **factless** fact has keys and no measures; you count its rows. A *coverage* factless
table records what was possible, such as which products were on promotion each day, so an anti-join can find
what was promoted but did not sell.

**Where it matters**
- Fulfilment and funnel reporting (time to ship, stuck orders): accumulating snapshots.
- Balances, inventory and subscriptions: periodic snapshots, because the level is the question.

**Watch out for**
- Forcing a level into a transaction fact and summing it later (see the stock example above).
- Treating an accumulating snapshot as append-only: each order must stay one row, updated in place.

### Example

Build an accumulating snapshot of order fulfilment: one row per order, a timestamp per milestone, and the lags
between them.

In [ ]:
con.sql("""
create table fact_order_fulfilment as
select order_id,
       channel,
       status,
       order_ts                                                   as placed_ts,
       shipped_ts,
       delivered_ts,
       date_diff('hour', order_ts, shipped_ts)                    as hours_to_ship,
       date_diff('hour', shipped_ts, delivered_ts)                as hours_in_transit,
       delivered_ts is null and status <> 'cancelled'             as is_open
from src_orders
""")
con.sql("select * exclude (placed_ts) from fact_order_fulfilment order by order_id").show()

In [ ]:
con.sql("""
select channel,
       count(*) filter (where delivered_ts is not null) as delivered,
       round(avg(hours_to_ship), 1)                     as avg_hours_to_ship,
       round(avg(hours_in_transit), 1)                  as avg_hours_in_transit,
       count(*) filter (where is_open)                  as still_open
from fact_order_fulfilment
group by channel order by channel
""").show()

**What to notice:** Store orders ship and arrive at the moment of sale (0 hours), while web orders average about 22 hours to ship and
76 in transit. Orders 1006 (shipped, not delivered) and 1008 (placed, not shipped) are `is_open`, with nulls in
the milestones they have not reached. When 1006 is delivered, its row is **updated**, not appended to; lesson 7
loads this table with `merge`.

### Example

A factless fact: promotions applied to orders. There is nothing to sum, so you count.

In [ ]:
con.sql("""
create table fact_order_promotion as
select op.order_id, op.promo_code, cast(o.order_ts as date) as order_date
from src_order_promotions op join src_orders o using (order_id);

select p.promo_code, p.promo_name, count(f.order_id) as orders_using_it
from src_promotions p left join fact_order_promotion f using (promo_code)
group by all order by orders_using_it desc, promo_code
""").show()

**What to notice:** `BULK` was used on two orders and the others on one each. The table records only that something happened; how
much revenue each promotion drove is a different question, and because order 1003 used two promotions, it needs
the allocation rules of lesson 4.

## Mixed grain: where order-level values go

Some measures exist at a coarser grain than the fact table. The shipping fee is charged once per order, not
per line. Copying it onto every line of the order puts two grains in one table, and summing it counts the fee
once per line.

There are three correct options:

1. **A separate fact at order grain** (`fact_order` with `shipping_fee`). Clean, and the usual choice.
2. **Allocate** the fee to lines by a stated rule (in proportion to line amount, say) so the line values add
   up to the order value exactly.
3. Keep it on one line only. This is fragile and confusing; avoid it.

**Where it matters**
- Order-level charges and discounts, taxes computed on the whole basket, invoice totals.
- Any time two sources at different grains are merged into one table.

**Watch out for**
- Allocation rounding: allocated cents must add back to the original amount, or totals drift.
- A column that is null on most rows and filled on one row per group: a sign of a hidden second grain.

### Example

Copy the fee onto lines, then compare with the order-level total and a proportional allocation.

In [ ]:
con.sql("""
with lines as (
    select f.order_id, f.line_no, f.net_amount, o.shipping_fee,
           f.net_amount / sum(f.net_amount) over (partition by f.order_id) as share
    from fact_order_line f join src_orders o using (order_id)
)
select (select sum(shipping_fee) from src_orders where status <> 'cancelled') as fee_at_order_grain,
       sum(shipping_fee)                                                    as fee_copied_to_lines,
       sum(round(shipping_fee * share, 2))                                  as fee_allocated_to_lines
from lines
""").show()

**What to notice:** The true fee total is 19.96: four web orders at 4.99. Copied to every line it becomes 34.93, because orders
1001 and 1005 have two and three lines. The proportional allocation is closer but still a cent out at 19.97:
order 1001's two equal lines each get 2.495, which rounds to 2.50, so that order's fee becomes 5.00. Production
allocation code gives the rounding remainder to one line (the largest, say) so each order adds back exactly.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** A stakeholder asks for "a table of orders with the products in them". What grain do you propose, and why?

**A.** One row per product on an order (order line). It answers product questions and still rolls up to orders by grouping on `order_id`; an order-grain table cannot be split back into products.

**Q2.** Your `fact_payment` should be one row per payment, but `count(*)` is 120 and `count(distinct payment_id)` is 118. What does that tell you?

**A.** The table does not have the grain it claims: two payments appear twice. Find the duplicate keys before using any total from it.

**Q3.** Is "number of active subscriptions" additive? How would you report it for a quarter?

**A.** It is semi-additive: it can be summed across plans or regions on one day, not across days. Report the value on the last day of the quarter, or the daily average.

**Q4.** Which kind of fact table fits "how long do refund requests wait at each approval step"?

**A.** An accumulating snapshot: one row per refund request with a timestamp per step and the lags between them, updated as the request moves.

**Q5.** Why is copying the order's shipping fee onto each order line wrong, even though every value is correct?

**A.** It mixes order grain into a line-grain table. Each value is right, but summing counts the fee once per line, so orders with more lines inflate the total.

## Key takeaways

- Pick a business process, declare the grain in one sentence, then choose dimensions and facts.
- Default to the atomic grain; aggregate later. The grain is the key, so check that it is unique.
- Fact-to-dimension joins must be many-to-one, so that totals do not change when you add a dimension.
- Sum additive measures freely, take closing or average values of semi-additive ones over time, and store the components of ratios.
- Transaction, periodic snapshot, accumulating snapshot and factless facts answer different questions.
- Keep each measure at its own grain: put order-level values on an order-grain fact or allocate them exactly.

**Next:** Continue with [Data modeling 2: star schema design](data-modeling-02-star-schema-design.ipynb).